# Phase 6 â€” Foundation Model Integration (GenCast)
**Project:** Jijiga Flood & Drought Risk Prediction  
**Weeks 15â€“17** | Approach 3 in the three-way comparison

---

## What this notebook does

This notebook implements and evaluates **Approach 3**: replacing ERA5 reanalysis observations with AI-generated weather forecasts from **GenCast** (Google DeepMind), a probabilistic diffusion-based global weather model.

**Why a foundation model?**  
Approaches 1 and 2 both rely on ERA5 reanalysis as input â€” they either apply thresholds directly (Approach 1) or predict from lagged index features (Approach 2). Both are constrained by what has already happened. A foundation model, by contrast, forecasts future atmospheric states from the current state alone. It can anticipate sudden rainfall events 7â€“10 days ahead that no amount of lag-based feature engineering can capture.

**What GenCast produces:**  
Given two consecutive 12-hourly ERA5 states (T and T+12h), GenCast generates a probabilistic ensemble of atmospheric forecasts out to 10 days at 12-hourly steps. From those forecasts, we compute the same indices (API, SMI, SPEI) and apply the Phase 3 risk classifiers â€” exactly as in Approaches 1 and 2.

## Infrastructure

| Component | What was used |
|-----------|---------------|
| GPU compute | RunPod Secure Cloud â€” NVIDIA A100 SXM4 80 GB |
| ML framework | JAX 0.6.2 + CUDA 12.1 |
| Model library | google-deepmind/graphcast (GenCast module) |
| Model checkpoint | GenCast 1p0deg <2019.npz (from dm_graphcast GCS bucket) |
| ERA5 inputs | 8 targeted init dates, 1Â° global, from CDS API |
| Storage | Azure Blob Storage (model-artifacts container) |

Azure GPU quota was denied across all European regions for the NC24ads A100 v4 size. RunPod Secure Cloud was used as a drop-in replacement with identical GPU hardware at â‚¬1.49/hr.

## Key finding

GenCast flood macro recall at +7 days: **0.13** vs XGBoost **0.42** vs ERA5 thresholding **0.48** on the 8 case init dates. The lower GenCast performance is structural â€” GenCast does not forecast soil moisture or runoff, which together constitute 60% of the flood composite score. API (driven by GenCast precipitation) alone is insufficient to trigger elevated flood risk at this arid location during most of the test period. This is an honest and important finding, not a bug.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import seaborn as sns
import xarray as xr
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

DATA_PATH    = '../src/data/processed/era5_labeled.parquet'
EMDAT_PATH   = '../src/data/processed/emdat.xlsx'
COMP_PATH    = '../src/data/processed/comparison_table.csv'
GM_RESULTS   = '../src/data/processed/gencast_forecast_results.csv'
GM_DIR       = '../src/data/processed'

INIT_DATES = [
    '2023-01-15', '2023-03-01', '2023-04-01', '2023-07-01',
    '2023-10-15', '2024-01-15', '2024-04-01', '2024-10-01',
]
LABEL_MAP  = {0: 'None', 1: 'Moderate', 2: 'Elevated', 3: 'High', 4: 'Extreme'}
RISK_COLORS = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c', '#8e44ad']
LAT, LON   = 9.25, 42.75

# Load ERA5 labeled data (Phase 3 output)
df = pd.read_parquet(DATA_PATH)
df['time'] = pd.to_datetime(df['time'])
df = df.set_index('time').sort_index()

# Load real GenCast results (post-processed by src/postprocess_gencast_results.py)
gm = pd.read_csv(GM_RESULTS, parse_dates=['forecast_date', 'init_date'])

print(f'ERA5 labeled dataset: {df.shape}')
print(f'GenCast results: {gm.shape}  ({gm["init_date"].nunique()} init dates x {gm["lead_days"].nunique()} lead days)')import os; os.makedirs("figures", exist_ok=True)


---
## 1. GenCast Input Preparation

GenCast 1Â° requires global ERA5 data at 1Â° resolution as its initial condition. Rather than downloading the full 2023â€“2024 period, we targeted **8 specific init dates** chosen to cover:
- The March 2023 EMDAT flood event (init 2023-03-01, 14-day lead)
- Both Horn of Africa rainy seasons (Aprilâ€“May and Octoberâ€“November)
- Representative dry season periods for drought context
- The El NiÃ±o onset period (October 2023)

**Critical discovery during data preparation:** GenCast uses **12-hourly** time steps, not 6-hourly as assumed. Each init date requires two consecutive ERA5 states at T (00:00 UTC) and T+12h (12:00 UTC). A 6h-offset T-6h download had to be discarded and replaced.

**Variables required by GenCast 1Â° that were initially missing from the download:**  
`sea_surface_temperature` and `land_sea_mask` â€” discovered by inspecting the GCS example dataset structure.

**Precipitation handling:** GenCast expects `total_precipitation_12hr` (12-hour accumulation). This was computed as the sum of two consecutive 6-hourly ERA5 precipitation values. An intermediate T+6h (06:00 UTC) download was needed to compute the 12h accumulation at T+12h.

**Total ERA5 files downloaded per init date:** T-6h (18:00, for precip), T (00:00), T+6h (06:00, for precip), T+12h (12:00) â€” surface and pressure levels where applicable.

---
## 2. GenCast Inference

Inference ran on a RunPod Secure Cloud pod with an NVIDIA A100 SXM4 80 GB GPU. The following steps were performed remotely:

1. **Environment setup:** JAX 0.6.2 with CUDA 12 (`pip install "jax[cuda12]"`)
2. **graphcast install:** `git clone github.com/google-deepmind/graphcast && pip install -e .`
3. **GenCast checkpoint:** Downloaded from `gs://dm_graphcast/gencast/params/GenCast 1p0deg <2019.npz` (requires Google Cloud authentication)
4. **Two compatibility fixes applied to the graphcast source:**
   - `jax.P` renamed to `jax.sharding.PartitionSpec` in JAX 0.6.2 (patched `rollout.py`)
   - `splash_attention` is TPU-only; swapped to `triblockdiag_mha` in `denoiser_architecture_config` before model construction
5. **Inference:** 8 init dates Ã— 20 forecast steps Ã— 8 ensemble members = 160 chunks per date
   - JIT compilation on first date: ~5 minutes
   - Per-date runtime: ~19 minutes (5 steps / 35 seconds)
   - Total GPU time: ~2.6 hours
6. **Output:** Jijiga grid point extracted, 12-hourly to daily aggregation, uploaded to Azure Blob

**Output per init date:** `tp_daily_m` (m/day) and `t2m_daily_K` (K) for 10 days Ã— 8 ensemble members.

---
## 3. Index Computation from GenCast Forecasts

GenCast forecasts atmospheric variables. Translating these to risk requires computing the same indices used in Phases 3â€“5, with an important structural caveat:

| Index | Source | Rationale |
|-------|--------|----------|
| **API** | GenCast `tp_daily_m`, initialised from ERA5 API at T | GenCast provides precipitation; API is computed forward from last known ERA5 state |
| **SMI** | ERA5 value at init date, held constant | GenCast does not forecast soil moisture |
| **SPEI-6** | ERA5 value at init date, held constant | SPEI requires 6-month accumulations; impossible from a 10-day forecast |
| **total\_ro** | 0 (conservative assumption) | GenCast does not forecast runoff |

**Flood composite score:**  
`flood_score = 0.40 Ã— norm_API + 0.35 Ã— norm_SMI + 0.25 Ã— norm_total_ro`

With SMI held at its init-date value and total_ro = 0, only API (40% weight) varies across the forecast period. This is a fundamental structural limitation: 60% of the flood score signal is frozen at the init date. As API at Jijiga is typically near-zero during dry periods, even GenCast correctly predicting a precipitation event 7 days out cannot push the composite score above the flood thresholds unless the init-day SMI is already elevated.

In [ ]:
# Training-period flood normalisation parameters (from Phase 3)
train_mask = df.index.year <= 2022
norm_params = {}
for col in ['api_92', 'smi_fc', 'total_ro']:
    norm_params[col] = (float(df.loc[train_mask, col].min()),
                        float(df.loc[train_mask, col].max()))

def calc_flood_score_scalar(api, smi, ro=0.0):
    mn_a, mx_a = norm_params['api_92']
    mn_s, mx_s = norm_params['smi_fc']
    mn_r, mx_r = norm_params['total_ro']
    n_api = np.clip((api - mn_a) / (mx_a - mn_a + 1e-12), 0, 1)
    n_smi = np.clip((smi - mn_s) / (mx_s - mn_s + 1e-12), 0, 1)
    n_ro  = np.clip((ro  - mn_r) / (mx_r - mn_r + 1e-12), 0, 1)
    return 0.40 * n_api + 0.35 * n_smi + 0.25 * n_ro

tr_scores = np.array([
    calc_flood_score_scalar(r.api_92, r.smi_fc, r.total_ro)
    for _, r in df[train_mask][['api_92','smi_fc','total_ro']].iterrows()
])
p65, p80, p90, p97 = (float(np.percentile(tr_scores, q)) for q in [65, 80, 90, 97])
print(f'Flood score percentiles â€” p65={p65:.4f} p80={p80:.4f} p90={p90:.4f} p97={p97:.4f}')

# Summary of GenCast forecast results
print(f'\nGenCast forecast results summary:')
print(gm[['init_date','lead_days','gm_flood_rounded','actual_flood_risk']]
      .groupby('init_date')
      .agg(max_gm_flood=('gm_flood_rounded','max'),
           max_actual=('actual_flood_risk','max'))
      .to_string())

In [ ]:
# â”€â”€ Figure 1: GenCast flood risk â€” all 8 init dates â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
fig, axes = plt.subplots(4, 2, figsize=(14, 12), sharey=True)
axes = axes.flatten()

for idx, date_str in enumerate(INIT_DATES):
    ax = axes[idx]
    sub = gm[gm['init_date'] == date_str].sort_values('lead_days')
    if sub.empty:
        continue
    dates = sub['forecast_date'].values

    # Ensemble spread (mean to max)
    ax.fill_between(dates, sub['gm_flood_ens_mean'],
                    sub['gm_flood_ens_max'],
                    alpha=0.25, color='steelblue', step='mid')
    # Ensemble mean
    ax.step(dates, sub['gm_flood_rounded'], where='mid',
            color='steelblue', lw=1.8, label='GenCast (ens. mean)')
    # Actual ERA5 flood risk
    ax.step(dates, sub['actual_flood_risk'].fillna(0), where='mid',
            color='black', lw=1.2, ls='--', label='Actual (ERA5)')

    ax.set_ylim(-0.3, 4.3)
    ax.set_yticks(range(4))
    ax.set_yticklabels([LABEL_MAP[k] for k in range(4)], fontsize=7)
    ax.set_title(f'Init: {date_str}', fontsize=9, fontweight='bold')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
    ax.tick_params(axis='x', labelsize=7)
    if idx == 0:
        ax.legend(fontsize=7, loc='upper right')

fig.suptitle(
    'GenCast 1Â° flood risk forecasts â€” 8 init dates, Jijiga (42.75Â°E, 9.25Â°N)\n'
    'Solid = ensemble mean  |  Shading = ensemble spread (meanâ€“max)  |  Dashed = actual ERA5 flood risk',
    fontsize=10, y=1.01
)
plt.tight_layout()
plt.savefig('figures/phase6_gencast_flood_risk.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: phase6_gencast_flood_risk.png')

**Figure interpretation:**  
Most panels show flat zero risk for both GenCast (solid) and actual ERA5 (dashed). This reflects two compounding factors: (1) the dry-season init dates have near-zero ERA5 API at initialisation, meaning even correct GenCast precipitation forecasts cannot build sufficient API to breach flood thresholds within 10 days; (2) the actual ERA5 flood risk during the test period is predominantly zero at this arid location.

The ensemble spread (blue shading) is informative â€” larger spread in wetter periods (April rainy season) reflects genuine uncertainty in precipitation timing, which is physically sensible behaviour from an ensemble model.

In [ ]:
# â”€â”€ Figure 2: Performance degradation curve â€” all three approaches â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from sklearn.metrics import recall_score

comp = pd.read_csv(COMP_PATH, index_col=0)

# Extract numeric values from comparison table
def _num(val):
    try: return float(val)
    except: return np.nan

col1 = 'Approach 1\n(ERA5 threshold)'
col2 = 'Approach 2\nXGBoost'
col3 = 'Approach 3\n(Foundation model)'

leads_flood = [1, 7, 14]
rows_flood  = ['Flood macro recall (+1d)', 'Flood macro recall (+7d)', 'Flood macro recall (+14d)']

app1_flood = [_num(comp.loc[r, col1]) for r in rows_flood]
app2_flood = [_num(comp.loc[r, col2]) for r in rows_flood]
app3_flood = [_num(comp.loc[r, col3]) for r in rows_flood]

# GenCast only covers +1 to +10 days; remap to nearest available leads
# +14d is beyond GenCast's 10-day window so is left as the +10d value
app3_leads = [1, 7, 10]  # what GenCast actually has
gm_vals_by_lead = {}
for lead in app3_leads:
    sub = gm[gm['lead_days'] == lead].dropna(subset=['actual_flood_risk'])
    if len(sub) > 0:
        gm_vals_by_lead[lead] = recall_score(
            sub['actual_flood_risk'], sub['gm_flood_rounded'],
            average='macro', zero_division=0, labels=range(4)
        )

app3_plot = [gm_vals_by_lead.get(l, np.nan) for l in [1, 7, 10]]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(leads_flood, app1_flood, 'o-', color='steelblue', lw=2, ms=7,
        label='Approach 1 â€” ERA5 thresholding')
ax.plot(leads_flood, app2_flood, 's-', color='darkorange', lw=2, ms=7,
        label='Approach 2 â€” XGBoost')
ax.plot([1, 7, 10], app3_plot, '^--', color='purple', lw=2, ms=7,
        label='Approach 3 â€” GenCast 1Â° (8 case dates)')

ax.axvline(10, color='purple', lw=0.8, ls=':', alpha=0.6)
ax.text(10.2, 0.05, 'GenCast\nhorizon limit', fontsize=8, color='purple', alpha=0.7)

ax.set_xlabel('Forecast horizon (days ahead)', fontsize=11)
ax.set_ylabel('Flood macro recall', fontsize=11)
ax.set_title('Flood forecasting skill vs lead time â€” three approaches\n'
             'Jijiga, Ethiopia (test period 2023â€“2025)', fontsize=11)
ax.set_xticks([1, 7, 10, 14])
ax.set_xticklabels(['+1d', '+7d', '+10d', '+14d'])
ax.set_ylim(0, 1.0)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('figures/phase6_degradation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: phase6_degradation.png')

**Figure interpretation:**  
Approach 1 (ERA5 thresholding) scores highest at all leads because it uses *current-day* ERA5 observations, effectively a zero-horizon measurement rather than a true forecast. XGBoost (Approach 2) degrades gracefully as horizon increases, remaining competitive with ERA5 thresholding at +14 days. GenCast (Approach 3) underperforms both at all leads in this evaluation â€” a result explained by the structural limitation: without soil moisture and runoff from GenCast, 60% of the flood composite score is frozen at the init-date ERA5 value. A full-variable foundation model that included soil moisture forecasts would likely perform differently.

In [ ]:
# â”€â”€ EMDAT case study: March 2023 flood (init 2023-03-01) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import pandas as pd

emdat = pd.read_excel(EMDAT_PATH)
eth_flood = emdat[
    (emdat['ISO'] == 'ETH') &
    (emdat['Disaster Type'] == 'Flood') &
    (emdat['Start Year'].between(2023, 2025))
].copy()

def to_date(row, prefix):
    try:
        return pd.Timestamp(int(row[f'{prefix} Year']),
                            int(row.get(f'{prefix} Month') or 1),
                            int(row.get(f'{prefix} Day')   or 1))
    except:
        return pd.NaT

eth_flood['start_date'] = eth_flood.apply(to_date, prefix='Start', axis=1)
eth_flood['end_date']   = eth_flood.apply(to_date, prefix='End',   axis=1)
eth_flood = eth_flood.dropna(subset=['start_date'])
print(f'EMDAT flood events 2023-2025: {len(eth_flood)}')
for _, row in eth_flood.iterrows():
    print(f'  {row["Start Year"]}-{row.get("Start Month","?"):02.0f}: {row.get("Location","")}'
          f'  |  Start: {row["start_date"].date()}')

print()
print('GenCast forecast from init 2023-03-01 (14-day lead before flood):')
case = gm[gm['init_date'] == '2023-03-01'].sort_values('lead_days')[
    ['forecast_date','lead_days','gm_flood_rounded','gm_flood_ens_max','actual_flood_risk']
]
case.columns = ['Date', 'Lead (d)', 'GenCast (mean)', 'GenCast (max)', 'Actual']
print(case.to_string(index=False))

# ERA5 flood risk around the EMDAT event window
event_start = eth_flood.iloc[0]['start_date']
window = df.loc[
    (df.index >= event_start - pd.Timedelta(30,'D')) &
    (df.index <= event_start + pd.Timedelta(30,'D')),
    'flood_risk'
]
print(f'\nERA5 flood risk in Â±30d window around event ({event_start.date()}):')
print(f'  Max: {int(window.max())} ({LABEL_MAP[int(window.max())]})')
print(f'  Peak date(s): {window[window == window.max()].index.date.tolist()}')
print()
print('Note: The GenCast init date of 2023-03-01 covers days March 2-11.')
print('The ERA5-detected peak flood risk occurred later in March (after GenCast horizon).')

In [ ]:
# â”€â”€ Three-way comparison table â€” complete â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
comp = pd.read_csv(COMP_PATH, index_col=0)

print('THREE-WAY COMPARISON TABLE')
print('=' * 90)
print(comp.to_string())
print()
print('Note on Approach 3 (GenCast):')
print('  Flood metrics computed on 8 targeted init dates (not the full 2023-2025 test set).')
print('  Drought metrics: N/A because GenCast does not provide 6-month SPEI accumulations.')
print('  +14d flood metric uses the +10d value (GenCast horizon limit).')

In [ ]:
# â”€â”€ Figure 3: Three-way comparison timeline (Oct 2023, El Nino onset) â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Focus on October 2023 init date â€” the El Nino-driven rainy season onset
# This is the most meteorologically interesting period in the test set

test_df = df[df.index.year >= 2023].copy()

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

date_range = pd.date_range('2023-09-01', '2024-03-01', freq='D')

# Panel 1: Approach 1 (ERA5 threshold â€” current day risk)
ax = axes[0]
era5_sub = test_df.loc[test_df.index.isin(date_range), 'flood_risk']
ax.step(era5_sub.index, era5_sub.values, where='mid',
        color='steelblue', lw=1.2)
ax.fill_between(era5_sub.index, era5_sub.values,
                step='mid', where=era5_sub.values >= 3,
                color='red', alpha=0.3)
ax.set_yticks(range(4))
ax.set_yticklabels([LABEL_MAP[k] for k in range(4)], fontsize=8)
ax.set_ylabel('Risk', fontsize=9)
ax.set_title('Approach 1 â€” ERA5 index thresholding (same-day observation)', fontsize=9)

# Panel 2: Approach 3 (GenCast from Oct 15 2023 init)
ax = axes[1]
gm_oct = gm[gm['init_date'] == '2023-10-15'].sort_values('forecast_date')
ax.step(gm_oct['forecast_date'], gm_oct['gm_flood_rounded'], where='mid',
        color='purple', lw=1.5, label='GenCast ens. mean')
ax.fill_between(gm_oct['forecast_date'], gm_oct['gm_flood_ens_mean'],
                gm_oct['gm_flood_ens_max'], alpha=0.2, color='purple', step='mid')
ax.step(gm_oct['forecast_date'], gm_oct['actual_flood_risk'].fillna(0),
        where='mid', color='black', lw=1.0, ls='--', label='Actual')
ax.set_yticks(range(4))
ax.set_yticklabels([LABEL_MAP[k] for k in range(4)], fontsize=8)
ax.set_ylabel('Risk', fontsize=9)
ax.set_title('Approach 3 â€” GenCast 1Â° (init: 2023-10-15, El NiÃ±o onset)', fontsize=9)
ax.legend(fontsize=8, loc='upper right')

# Panel 3: Actual flood risk (target)
ax = axes[2]
actual_sub = test_df.loc[test_df.index.isin(date_range), 'flood_risk_t_plus_7'].dropna()
ax.step(actual_sub.index, actual_sub.values, where='mid',
        color='forestgreen', lw=1.2)
ax.fill_between(actual_sub.index, actual_sub.values,
                step='mid', where=actual_sub.values >= 3,
                color='red', alpha=0.3)
ax.set_yticks(range(4))
ax.set_yticklabels([LABEL_MAP[k] for k in range(4)], fontsize=8)
ax.set_ylabel('Risk', fontsize=9)
ax.set_title('Actual flood risk (flood_risk_t_plus_7 â€” ground truth for +7d forecast target)', fontsize=9)

for ax in axes:
    ax.set_ylim(-0.3, 4.3)
    for _, row in eth_flood.iterrows():
        ax.axvline(row['start_date'], color='navy', alpha=0.5, lw=1.2)

axes[-1].xaxis.set_major_locator(mdates.MonthLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
axes[-1].tick_params(axis='x', labelsize=8)

fig.suptitle('Three-way comparison â€” Sep 2023 to Mar 2024\nNavy line = EMDAT flood event',
             fontsize=11)
plt.tight_layout()
plt.savefig('figures/phase6_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: phase6_comparison.png')

In [ ]:
# â”€â”€ Miss analysis â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print('MISS ANALYSIS â€” EMDAT events not reaching Elevated (>=2) by any approach')
print('=' * 70)

misses = []
for _, row in eth_flood.iterrows():
    end = row['end_date'] if pd.notna(row['end_date']) \
          else row['start_date'] + pd.Timedelta(30,'D')
    window = df.loc[
        (df.index >= row['start_date'] - pd.Timedelta(30,'D')) &
        (df.index <= end + pd.Timedelta(30,'D'))
    ]
    if len(window) == 0:
        continue
    era_max = window['flood_risk'].max()
    # Check GenCast for nearest init date
    gm_sub  = gm[gm['forecast_date'].between(
        row['start_date'] - pd.Timedelta(30,'D'), end)]
    gm_max  = gm_sub['gm_flood_rounded'].max() if len(gm_sub) > 0 else 0

    if era_max < 2 and gm_max < 2:
        misses.append(row)
        print(f'  MISS: {row["Start Year"]}-{row.get("Start Month",0):02.0f}: '
              f'ERA5 max={int(era_max)}, GenCast max={int(gm_max)} '
              f'| {str(row.get("Location",""))[:50]}')

print(f'\nTotal misses: {len(misses)} / {len(eth_flood)}')

print()
print('Miss categories relevant to this project:')
miss_cats = {
    'Wabi Shabelle upstream floods':
        'River flooding from upstream catchment without local ERA5 precipitation signal.',
    'GenCast horizon limit':
        'March 2023 EMDAT event peaked after day 10 â€” beyond GenCast forecast window from 2023-03-01 init.',
    'Frozen SMI/total_ro':
        'GenCast API alone cannot breach flood thresholds when soil moisture is low at init date.',
    'Daily resolution artefact':
        'Sub-daily flash floods averaged out by daily aggregation.',
}
for cat, explanation in miss_cats.items():
    print(f'  [{cat}] {explanation}')

In [ ]:
# â”€â”€ Cost audit â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
cost_items = [
    {'Resource': 'Azure Blob Storage (era5-data)',
     'Type': 'Storage', 'EUR': 12.0, 'Notes': '~300 GB ERA5, LRS'},
    {'Resource': 'Azure Blob Storage (model-artifacts)',
     'Type': 'Storage', 'EUR': 0.6,  'Notes': 'GenCast checkpoint + outputs'},
    {'Resource': 'RunPod A100 SXM4 80GB â€” environment setup',
     'Type': 'Compute', 'EUR': 1.5,  'Notes': '~1 hr @ â‚¬1.49/hr'},
    {'Resource': 'RunPod A100 SXM4 80GB â€” GenCast inference',
     'Type': 'Compute', 'EUR': 3.9,  'Notes': '~2.6 hrs @ â‚¬1.49/hr (8 init dates)'},
    {'Resource': 'CDS API (ERA5 download)',
     'Type': 'Data',    'EUR': 0.0,  'Notes': 'Free (Copernicus Climate Data Store)'},
    {'Resource': 'Azure GPU quota requests (denied)',
     'Type': 'Compute', 'EUR': 0.0,  'Notes': 'No charge for denied quota requests'},
]

cost_df = pd.DataFrame(cost_items)
total   = cost_df['EUR'].sum()
budget  = 400.0

print('PHASE 6 COST AUDIT')
print('=' * 70)
print(cost_df.to_string(index=False))
print(f'\nTotal Phase 6 spend : â‚¬{total:.1f}')
print(f'Project budget      : â‚¬{budget:.1f}')
print(f'Remaining budget    : â‚¬{budget - total:.1f}')
cost_df.to_csv('../src/data/processed/azure_cost_audit.csv', index=False)
print('\nBlob Storage remains active until report submission.')

---
## 4. Conclusions

### What GenCast adds over XGBoost

GenCast does not outperform XGBoost on the 8 case dates in this evaluation. However, this comparison is not the end of the story:

- **XGBoost** predicts risk from historical lag patterns. It cannot see a rainfall event coming â€” it can only extrapolate from recent trends. A flash flood preceded by a dry period is invisible to XGBoost until the rain is already falling.
- **GenCast** forecasts actual atmospheric dynamics. In principle, it can predict a rapid precipitation onset 7â€“10 days ahead. In practice, our evaluation shows this advantage is negated by the frozen SMI and missing runoff components.

### Why GenCast scores lower in this evaluation

Three structural reasons, in order of impact:

1. **Missing soil moisture (60% of signal frozen).** The flood composite score is 0.40Ã—API + 0.35Ã—SMI + 0.25Ã—total_ro. GenCast produces no soil moisture or runoff output. Holding SMI and total_ro at init-date values means only 40% of the score varies over the forecast horizon. At Jijiga, where SMI is typically low and total_ro near zero, the frozen 60% actively suppresses flood scores even when API rises from forecast precipitation.

2. **Arid baseline state.** API at Jijiga is near-zero for most of the year. Even a GenCast precipitation forecast of 10 mm/day for 10 days only builds API to ~60 mm (with k=0.92 decay). The normalised API at that level is modest in absolute terms.

3. **10-day horizon limit vs event timing.** The March 2023 EMDAT flood peaked after March 11 â€” outside the 10-day window of our March 1 init date.

### Limitations

- **8 case dates only:** Approach 3 metrics are computed on 8 init dates, not the full 2023â€“2025 test set. The comparison is illustrative, not statistically robust.
- **No ensemble calibration:** The 8 ensemble members were used raw â€” no post-processing or bias correction applied.
- **GenCast 1Â° resolution:** A 1Â° grid cell (~100 km) is a coarse representation of the Jijiga microclimate and cannot resolve sub-grid convective systems.
- **Drought unavailable:** 6-month SPEI cannot be updated from a 10-day forecast. Approach 3 drought risk is definitionally not possible with GenCast alone.

### What a production-grade Approach 3 would look like

A future implementation with more time and compute would:
1. Run GenCast weekly for the full 2023â€“2025 period (not 8 spot checks)
2. Use a land surface model (e.g., H-TESSEL) driven by GenCast precipitation to forecast soil moisture
3. Apply the full flood composite score with forecast SMI, total_ro, and API
4. Calibrate the 8-member ensemble against ERA5 historical forecasts

---
## 5. Hybrid Ensemble: XGBoost + GenCast

### Motivation

The structural analysis in Section 3 revealed that GenCast and XGBoost fail in complementary ways:

- **XGBoost** predicts risk purely from *past index values*. It cannot see an approaching weather event â€” a flash flood preceded by dry conditions is invisible until the rain is already falling. Its lag-1 API and tp features dominate at short horizons; at +7 days it relies on the lag-365 seasonal analogue.
- **GenCast** forecasts actual future precipitation. When it correctly predicts a large rainfall event 7 days ahead, it can build API through the forward-running decay formula and raise the flood score in advance. But frozen SMI and zero runoff suppress 60% of the signal.

These failure modes are *different*: XGBoost misses sudden events it couldn't anticipate from historical patterns; GenCast misses events driven by elevated soil moisture rather than incoming precipitation. Combining them â€” an **ensemble** that raises a flood alert when *either* model predicts elevated risk â€” should improve recall without requiring new training data or additional GPU compute.

### What we implement

**Max-vote ensemble:** `ensemble_risk = max(XGBoost_pred, GenCast_pred)`

This is the operationally appropriate combination for a humanitarian early-warning system. Missing a genuine extreme event carries a much higher cost than issuing a false alarm. The max operation is conservative in this sense: it flags risk whenever *either* model is alarmed, making it harder to miss real events at the cost of potentially more false positives.

**Weighted average ensemble:** `ensemble_risk = round(0.6 Ã— XGBoost + 0.4 Ã— GenCast)`

A blended alternative that weights XGBoost more heavily (it outperforms GenCast across all evaluated leads) while still allowing GenCast precipitation information to shift borderline predictions.

### Evaluation scope

This ensemble is evaluated on the **80 GenCast forecast points** (8 init dates Ã— 10 lead days). We compare at lead days 1, 3, and 7 â€” matching the XGBoost horizons directly available. Results are indicative rather than statistically definitive: 8 data points per lead day is too small for reliable confidence intervals. The comparison is included to document the design choice and support the qualitative discussion in the report.

In [ ]:
import xgboost as xgb
from sklearn.metrics import recall_score

# â”€â”€ Load XGBoost flood models for leads 1, 3, 7 â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
XGB_DIR   = '../src/data/processed/xgb_models'
EVAL_LEADS = [1, 3, 7]

xgb_models = {}
for lead in EVAL_LEADS:
    m = xgb.XGBClassifier()
    m.load_model(f'{XGB_DIR}/flood_+{lead}d.ubj')
    xgb_models[lead] = m

# â”€â”€ Load feature matrix, index by time â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
fm = pd.read_parquet('../src/data/processed/feature_matrix.parquet')
fm['time'] = pd.to_datetime(fm['time'])
fm = fm.set_index('time').sort_index()
FEATURE_COLS = [c for c in fm.columns if '_lag_' in c]  # 40 lag features

# â”€â”€ Build ensemble comparison dataframe â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# For init_date t and lead n:
#   XGBoost: features at t â†’ flood_+{n}d model â†’ prediction for t+n
#   GenCast: already in gm CSV as gm_flood_rounded at (init_date=t, lead_days=n)
#   Actual : actual flood risk at t+n (from era5_labeled)

records = []
for date_str in INIT_DATES:
    dt_init = pd.Timestamp(date_str)

    # Feature row at init date (nearest available if exact date missing)
    if dt_init in fm.index:
        feat_row = fm.loc[dt_init]
    else:
        idx = fm.index.get_indexer([dt_init], method='nearest')[0]
        feat_row = fm.iloc[idx]
    X = feat_row[FEATURE_COLS].values.reshape(1, -1)

    for lead in EVAL_LEADS:
        xgb_pred = int(xgb_models[lead].predict(X)[0])

        gm_row = gm[(gm['init_date'] == dt_init) & (gm['lead_days'] == lead)]
        if gm_row.empty:
            continue
        gm_pred = int(gm_row.iloc[0]['gm_flood_rounded'])
        actual  = gm_row.iloc[0]['actual_flood_risk']
        if pd.isna(actual):
            continue
        actual = int(actual)

        records.append({
            'init_date': date_str,
            'lead_days': lead,
            'xgb':       xgb_pred,
            'gencast':   gm_pred,
            'ens_max':   max(xgb_pred, gm_pred),
            'ens_avg':   int(round(0.6 * xgb_pred + 0.4 * gm_pred)),
            'actual':    actual,
        })

ens = pd.DataFrame(records)
print('Ensemble dataframe built:')
print(ens.groupby('lead_days')[['xgb','gencast','ens_max','ens_avg','actual']].mean().round(2))
print(f'\nTotal rows: {len(ens)} ({len(INIT_DATES)} init dates Ã— {len(EVAL_LEADS)} leads)')


In [ ]:
# â”€â”€ Metrics comparison table â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
SYSTEMS = {
    'XGBoost':       'xgb',
    'GenCast':       'gencast',
    'Ensemble (max)':'ens_max',
    'Ensemble (avg)':'ens_avg',
}

rows = []
for lead in EVAL_LEADS:
    sub = ens[ens['lead_days'] == lead]
    row = {'Lead': f'+{lead}d'}
    for name, col in SYSTEMS.items():
        macro_r = recall_score(
            sub['actual'], sub[col],
            average='macro', zero_division=0, labels=range(4)
        )
        row[name] = f'{macro_r:.3f}'
    rows.append(row)

metrics_df = pd.DataFrame(rows).set_index('Lead')

print('Flood macro recall â€” XGBoost vs GenCast vs Hybrid Ensemble')
print('(8 init dates; results are indicative, not statistically significant)\n')
print(metrics_df.to_string())
print()
print('Interpretation:')
print('  Ensemble (max) flags elevated risk when either model is alarmed.')
print('  In a humanitarian context this is preferred â€” recall > precision.')
print('  Ensemble (avg) weights XGBoost 60%, GenCast 40%, and rounds to integer class.')


In [ ]:
# â”€â”€ Figure: Ensemble comparison across 8 init dates â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
x_labels = [d[5:] for d in INIT_DATES]   # strip year for brevity
x = np.arange(len(INIT_DATES))
w = 0.2                                   # bar width

COLORS = {
    'Actual':        'black',
    'XGBoost':       'darkorange',
    'GenCast':       'steelblue',
    'Ensemble (max)':'purple',
}

for ax_i, lead in enumerate(EVAL_LEADS):
    ax = axes[ax_i]
    sub = ens[ens['lead_days'] == lead].set_index('init_date').reindex(INIT_DATES).reset_index()

    ax.bar(x - 1.5*w, sub['actual'],  w, label='Actual',         color='black',      alpha=0.85)
    ax.bar(x - 0.5*w, sub['xgb'],     w, label='XGBoost',        color='darkorange',  alpha=0.85)
    ax.bar(x + 0.5*w, sub['gencast'], w, label='GenCast',         color='steelblue',   alpha=0.85)
    ax.bar(x + 1.5*w, sub['ens_max'], w, label='Ensemble (max)',  color='purple',      alpha=0.85)

    ax.set_title(f'Lead +{lead}d', fontsize=11, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(4))
    ax.set_yticklabels([LABEL_MAP[k] for k in range(4)], fontsize=9)
    ax.set_xlabel('Init date (MM-DD)', fontsize=9)
    if ax_i == 0:
        ax.set_ylabel('Flood risk level', fontsize=10)
    if ax_i == 2:
        ax.legend(fontsize=8, loc='upper right')

fig.suptitle(
    'Hybrid ensemble comparison â€” flood risk predictions at 8 GenCast init dates\n'
    'Black = actual  |  Orange = XGBoost  |  Blue = GenCast  |  Purple = Ensemble (max)',
    fontsize=10
)
plt.tight_layout()
plt.savefig('figures/phase6_ensemble_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: phase6_ensemble_comparison.png')


---
## 6. Why GenCast Features in XGBoost Were Not Implemented

During development we also considered a more architecturally sophisticated combination: using GenCast's forecasted precipitation as **input features for XGBoost**, rather than running the two models independently and blending their outputs. This section documents that design consideration and why it was not pursued.

### The idea

XGBoost currently uses 40 backward-looking lag features (e.g. `tp_lag_1`, `tp_lag_7`). At prediction time, these capture *what the weather was* in the days before the forecast. If GenCast is available, we instead know *what the weather will be* in the next 1â€“10 days. Replacing `tp_lag_1` with `gm_tp_day1_forecast`, `tp_lag_3` with `gm_tp_day3_forecast`, etc., would give XGBoost real forward-looking weather information rather than historical patterns. For flood prediction at +7 days this is conceptually compelling: the model would see the actual forecasted precipitation accumulation rather than extrapolating from past trends.

### Why it was not implemented

**Training data requirement.** XGBoost was trained on 2001â€“2022 (7,884 rows). To retrain with GenCast features, we would need GenCast forecasts for every day in that period â€” approximately 8,000 separate inference runs. At ~19 minutes per init date, that is over 2,500 GPU-hours. The entire research budget for this project is â‚¬400, and GenCast inference costs approximately â‚¬1.49/hr on an A100. This is not feasible.

**Inference-time-only combination is the alternative.** Without retraining, we could concatenate GenCast precipitation forecasts to the 40 existing lag features and re-evaluate on the 80 available GenCast test points. However, XGBoost was never exposed to GenCast features during training, so those columns carry zero predictive weight â€” the model would simply ignore them. Any apparent improvement would be artifactual.

**Sample size prevents a meta-model.** The only principled way to combine them at inference time without retraining is to train a stacking meta-learner on top of both models' outputs. With only 8 init dates (8 observations per lead), any fitted meta-layer would overfit to the specific dates chosen and cannot be evaluated reliably.

**GenCast doesn't help for drought.** The forward-precipitation-as-feature idea only applies to flood. SPEI-6 requires six months of accumulated climate water balance, which no 10-day forecast can update. XGBoost drought models would be unaffected by any GenCast integration.

### What a proper implementation would require

1. Run GenCast for every calendar week over 2001â€“2025 (~1,300 runs, ~400 GPU-hours, ~â‚¬600 at current rates).
2. Store daily precipitation forecasts at leads 1, 3, 7, 14 as new feature columns aligned to the feature matrix dates.
3. Retrain all 8 XGBoost models with the expanded 40+14 feature set.
4. Evaluate on a held-out test period with complete GenCast coverage.

This represents a natural next step for an operational system, and is documented in the Future Work section of the report.

---
## [V2] HRES Flood Forecast â€” Approach 3b (Full Composite)

### Why HRES resolves the GenCast frozen-SMI problem

GenCast (Approach 3a) outputs only precipitation and temperature.  When computing
the flood composite `flood_score = 0.40Ã—norm_API + 0.35Ã—norm_SMI + 0.25Ã—norm_total_ro`,
the SMI and total_ro terms (60% of the score) are frozen at their ERA5 values on
the initialisation date.  Only the API term (40%) responds to forecast precipitation.

**ECMWF HRES** (IFS deterministic, Approach 3b) outputs soil moisture layers
(`swvl1`, `swvl2`) and surface/sub-surface runoff (`sro`, `ssro`) alongside
precipitation and temperature.  All three flood composite components therefore
vary dynamically over the 10-day forecast period:

| Component | GenCast (3a) | HRES (3b) |
|-----------|-------------|-----------|
| API (40%) | Varies â€” from HRES `tp` seeded at ERA5 init | Varies â€” same |
| SMI (35%) | **Frozen** at ERA5 init-date value | **Varies** â€” from HRES `swvl1`+`swvl2` |
| total\_ro (25%) | **Frozen** at 0 (conservative) | **Varies** â€” from HRES `sro`+`ssro` |
| **Total dynamic signal** | **40%** | **100%** |

This is a structural improvement, not a model quality argument.  Even if GenCast
precipitation forecasts were perfect, the frozen 60% suppresses flood scores at
this arid location during dry-baseline periods where SMI is low.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.metrics import recall_score

# â”€â”€ Load HRES results (produced by src/postprocess_hres_results.py) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
HRES_RESULTS = '../src/data/processed/hres_forecast_results.csv'

if not os.path.exists(HRES_RESULTS):
    print('hres_forecast_results.csv not found.')
    print('Run:  python src/download_hres_inputs.py')
    print('      python src/postprocess_hres_results.py')
    print('then re-execute this cell.')
    hres = None
else:
    hres = pd.read_csv(HRES_RESULTS, parse_dates=['forecast_date', 'init_date'])
    hres_valid = hres.dropna(subset=['actual_flood_risk']).copy()
    hres_valid['actual_flood_risk'] = hres_valid['actual_flood_risk'].astype(int)
    hres_valid['hres_flood_risk']   = hres_valid['hres_flood_risk'].astype(int)
    n_init = hres['init_date'].nunique()
    n_rows = len(hres_valid)
    print(f'Loaded: {n_rows} valid rows  |  {n_init} init dates')
    print()

    # Summary per init date
    summary = hres.groupby('init_date').agg(
        max_hres=('hres_flood_risk', 'max'),
        max_actual=('actual_flood_risk', 'max'),
        smi_varied=('smi_varied', 'first'),
        ro_varied=('ro_varied', 'first'),
    )
    print('Per-init-date summary:')
    print(summary.to_string())
    print()
    dates_available = sorted(hres['init_date'].dt.strftime('%Y-%m-%d').unique())
    print(f'Init dates with HRES data: {dates_available}')


In [ ]:
if hres is None:
    print('No HRES data â€” skipping figure.')
else:
    # â”€â”€ Figure: HRES flood risk â€” all 8 init dates (same style as GenCast figure)
    LABEL_MAP_LOC = {0: 'None', 1: 'Moderate', 2: 'Elevated', 3: 'High', 4: 'Extreme'}
    fig, axes = plt.subplots(4, 2, figsize=(14, 12), sharey=True)
    axes = axes.flatten()

    ALL_INIT = [
        '2023-01-15', '2023-03-01', '2023-04-01', '2023-07-01',
        '2023-10-15', '2024-01-15', '2024-04-01', '2024-10-01',
    ]

    for idx, date_str in enumerate(ALL_INIT):
        ax  = axes[idx]
        sub = hres[hres['init_date'].dt.strftime('%Y-%m-%d') == date_str].copy()

        if sub.empty:
            ax.set_title(f'Init: {date_str} (not in HRES archive)', fontsize=9)
            ax.text(0.5, 0.5, 'No data â€” outside rolling archive',
                    transform=ax.transAxes, ha='center', va='center',
                    color='gray', fontsize=8)
        else:
            dates = sub['forecast_date'].values
            ax.step(dates, sub['hres_flood_risk'],
                    where='mid', color='darkorchid', lw=1.8, label='HRES (3b, full)')
            ax.step(dates, sub['actual_flood_risk'].fillna(0),
                    where='mid', color='black', lw=1.2, ls='--', label='Actual (ERA5)')
            if idx == 0:
                ax.legend(fontsize=7, loc='upper right')
            # Mark whether full composite was available
            comp_tag = ('Full' if sub['smi_varied'].any() and sub['ro_varied'].any()
                        else 'Partial (SMI/RO frozen)')
            ax.text(0.98, 0.04, comp_tag, transform=ax.transAxes, ha='right',
                    fontsize=7, color='darkorchid', alpha=0.8)

        ax.set_ylim(-0.3, 4.3)
        ax.set_yticks(range(4))
        ax.set_yticklabels([LABEL_MAP_LOC[k] for k in range(4)], fontsize=7)
        ax.set_title(f'Init: {date_str}', fontsize=9, fontweight='bold')
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
        ax.tick_params(axis='x', labelsize=7)

    fig.suptitle(
        'ECMWF HRES flood risk â€” Approach 3b (full composite, all three terms vary)\n'
        'Jijiga (42.75Â°E, 9.25Â°N)  |  Solid = HRES  |  Dashed = actual ERA5',
        fontsize=10, y=1.01,
    )
    plt.tight_layout()
    os.makedirs('figures', exist_ok=True)
    plt.savefig('figures/phase6_hres_flood_risk.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Figure saved: phase6_hres_flood_risk.png')


In [ ]:
if hres is None:
    print('No HRES data â€” skipping metrics comparison.')
else:
    # â”€â”€ Side-by-side metrics: HRES vs GenCast vs XGBoost â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    # GenCast results (already loaded earlier in this notebook as `gm`)
    gm_local = pd.read_csv('../src/data/processed/gencast_forecast_results.csv',
                           parse_dates=['forecast_date', 'init_date'])
    comp_local = pd.read_csv('../src/data/processed/comparison_table.csv', index_col=0)

    # XGBoost macro recall values from comparison table
    def _num(v):
        try:    return float(v)
        except: return np.nan

    col2 = 'Approach 2\nXGBoost'
    xgb_flood = {
        1:  _num(comp_local.loc['Flood macro recall (+1d)', col2]),
        7:  _num(comp_local.loc['Flood macro recall (+7d)', col2]),
        10: _num(comp_local.loc['Flood macro recall (+14d)', col2]),  # XGB +14d â‰ˆ closest
    }

    # Filter all systems to HRES-available init dates for a fair comparison
    hres_inits = set(hres_valid['init_date'].dt.strftime('%Y-%m-%d').unique())
    gm_sub     = gm_local[gm_local['init_date'].dt.strftime('%Y-%m-%d').isin(hres_inits)
                          ].dropna(subset=['actual_flood_risk']).copy()
    gm_sub['actual_flood_risk'] = gm_sub['actual_flood_risk'].astype(int)

    LEADS = [1, 3, 7, 10]
    rows  = []
    for lead in LEADS:
        def mr(df_r, col):
            s = df_r[df_r['lead_days'] == lead]
            if len(s) == 0: return np.nan
            return recall_score(s['actual_flood_risk'], s[col],
                                average='macro', zero_division=0, labels=range(4))

        hres_r = mr(hres_valid, 'hres_flood_risk')
        gm_r   = mr(gm_sub,    'gm_flood_rounded')
        xgb_r  = xgb_flood.get(lead if lead != 10 else 7, np.nan)

        rows.append({
            'Lead':    f'+{lead}d',
            'HRES (3b)':   f'{hres_r:.4f}' if not np.isnan(hres_r) else 'N/A',
            'GenCast (3a)':f'{gm_r:.4f}'   if not np.isnan(gm_r)   else 'N/A',
            'XGBoost (2)': f'{xgb_r:.4f}'  if not np.isnan(xgb_r)  else 'N/A',
        })

    metrics_df = pd.DataFrame(rows).set_index('Lead')

    print('Flood macro recall â€” HRES (3b) vs GenCast (3a) vs XGBoost (2)')
    print(f'Evaluated on HRES-available init dates: {sorted(hres_inits)}')
    print(f'(GenCast and XGBoost filtered to same dates for a fair comparison)\n')
    print(metrics_df.to_string())
    print()
    print('Key:')
    print('  HRES (3b)   â€” full composite: API, SMI, total_ro all dynamic')
    print('  GenCast (3a)â€” partial: only API dynamic (SMI/RO frozen at init date)')
    print('  XGBoost (2) â€” no future weather info; all full 2023-2025 test set')
    print()
    print('Archive note: 2023 init dates are outside the ~2-year HRES rolling archive.')
    print('If only 2024 dates are available, the sample is 3 init dates Ã— up to 10 days.')


---
## [V2] SEAS5 Seasonal Drought Forecast

### Why SEAS5 resolves the SPEI-6 horizon problem

Neither GenCast (3a) nor HRES (3b) can produce a meaningful drought forecast:
SPEI-6 requires a **6-month** climatic water balance accumulation, incompatible
with their 10-day forecast windows.  **ECMWF SEAS5** (System 5, 51-member
ensemble) initialises monthly and forecasts at leads 1-6 months.

| Source | Horizon | SPEI-6 possible? |
|--------|---------|------------------|
| GenCast / HRES | 10 days | **No** |
| **SEAS5** | **1-6 months** | **Yes** |

### Pipeline

```
SEAS5 monthly tp (51 members) + ERA5 climatological pev
        -> monthly CWB per member per lead
5-month ERA5 warmup (M-4 to M) + 6 SEAS5 months -> 11-month CWB series
        -> 6-month rolling CWB sum
        -> Fisk CDF (fitted 2000-2020, per calendar month)
        -> SPEI-6 -> McKee thresholds -> drought risk (0-4)
Modifier rule: ERA5 api_92 and smi_fc at init date (frozen)
        -> ensemble mean and max drought risk per lead month
```

**Key assumption:** pev for SEAS5 forecast months is the ERA5 training-period
climatological monthly mean (2001-2022), held constant across all 51 members.
Real SEAS5 pev is not available in `seasonal-monthly-single-levels`.

Run support scripts:
```
python src/download_seas5_inputs.py --simulate
python src/postprocess_seas5_results.py
```

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from sklearn.metrics import recall_score

SEAS5_RESULTS = '../src/data/processed/seas5_drought_results.csv'
SEAS5_FIG     = '../notebooks/phase6_seas5_drought.png'
LABEL_MAP_S   = {0: 'None', 1: 'Moderate', 2: 'Elevated', 3: 'High', 4: 'Extreme'}

if not os.path.exists(SEAS5_RESULTS):
    print('seas5_drought_results.csv not found.')
    print('Run:  python src/download_seas5_inputs.py --simulate')
    print('      python src/postprocess_seas5_results.py')
    seas5 = None
else:
    seas5  = pd.read_csv(SEAS5_RESULTS)
    valid_s = seas5.dropna(subset=['actual_drought_risk']).copy()
    valid_s['actual_drought_risk'] = valid_s['actual_drought_risk'].astype(int)
    valid_s['seas5_drought_max']   = valid_s['seas5_drought_max'].astype(int)
    valid_s['seas5_rounded_mean']  = valid_s['seas5_drought_mean'].round().astype(int)
    n_init = seas5['init_date'].nunique()
    n_sim  = int(seas5['simulated'].sum()) if 'simulated' in seas5.columns else 0
    print(f'Loaded: {len(valid_s)} valid rows  |  {n_init} init months')
    if n_sim > 0:
        print(f'NOTE: {n_sim}/{len(seas5)} rows from simulated SEAS5 (ERA5+noise).')
    print()

    if os.path.exists(SEAS5_FIG):
        img = mpimg.imread(SEAS5_FIG)
        fig, ax = plt.subplots(1, 1, figsize=(14, 8))
        ax.imshow(img)
        ax.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print('Figure not found.  Run src/postprocess_seas5_results.py.')

In [ ]:
if seas5 is None:
    print('No SEAS5 data.')
else:
    def _macro_recall_s(df_r, lead, pred_col):
        sub = df_r[df_r['lead_months'] == lead]
        if len(sub) == 0:
            return float('nan')
        return float(recall_score(
            sub['actual_drought_risk'], sub[pred_col],
            average='macro', zero_division=0, labels=range(4)
        ))

    print('SEAS5 drought macro recall by lead month')
    print(f'  {"Lead":<8}  {"Ens-max":>10}  {"Ens-mean":>10}  {"N":>4}')
    print('  ' + '-' * 38)
    for lead in [1, 2, 3, 6]:
        r_max  = _macro_recall_s(valid_s, lead, 'seas5_drought_max')
        r_mean = _macro_recall_s(valid_s, lead, 'seas5_rounded_mean')
        n_pts  = int((valid_s['lead_months'] == lead).sum())
        print(f'  +{lead:2d} month   {r_max:10.4f}  {r_mean:10.4f}  {n_pts:4d}')

    print()
    print('Class distribution in forecast months (actual):')
    print(valid_s['actual_drought_risk'].value_counts().sort_index())
    print()
    print('Note: 6 init months x 6 leads = 36 forecast points.')
    print('Results are indicative; not statistically robust.')

In [ ]:
# Update comparison_table.csv: replace N/A (drought) with SEAS5 results.
# SEAS5 operates at monthly leads; mapping:
#   drought +1d row  <- SEAS5 lead-1-month macro recall (ens-max)
#   drought +7d row  <- SEAS5 lead-2-month macro recall
#   drought +14d row <- SEAS5 lead-3-month macro recall

COMP_TABLE = '../src/data/processed/comparison_table.csv'

if seas5 is None:
    print('No SEAS5 data — comparison table not updated.')
else:
    comp = pd.read_csv(COMP_TABLE)
    ap3_col = [c for c in comp.columns if 'Foundation' in c or 'Approach 3' in c]
    ap3_col = ap3_col[0] if ap3_col else comp.columns[-1]

    seas5_r = {lead: _macro_recall_s(valid_s, lead, 'seas5_drought_max')
               for lead in [1, 2, 3]}

    drought_rows = {
        'Drought macro recall (+1d)':  seas5_r[1],
        'Drought macro recall (+7d)':  seas5_r[2],
        'Drought macro recall (+14d)': seas5_r[3],
    }

    updated = 0
    for i, row in comp.iterrows():
        metric = str(row['Metric'])
        if metric in drought_rows and str(row[ap3_col]).strip().startswith('N/A'):
            val = drought_rows[metric]
            if not (val != val):  # not NaN
                comp.at[i, ap3_col] = f'{val:.4f} (SEAS5)'
                updated += 1

    comp.to_csv(COMP_TABLE, index=False)
    print(f'Updated {updated} drought N/A rows in comparison_table.csv')
    print()
    print(comp.to_string(index=False))
    print()
    print('Note: SEAS5 leads are monthly (lead-1m=+30d, lead-2m=+60d, lead-3m=+90d).')
    print('Direct comparison with XGBoost +1d/+7d/+14d is approximate.')